# Ch 16 — perplexity 너머

PPL 이 무엇이고, 언제 거짓말하는지, 그리고 그 대신 무엇을 봐야 하는지 배웁니다.

In [ ]:
# 필요 시 설치
# !pip install torch tokenizers

import math
import random
import torch
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'device: {device}')

## 1. PPL 식 한 줄

$$
\text{PPL}(x_1 \dots x_T) = \exp\!\left(-\frac{1}{T} \sum_{t=1}^T \log p(x_t \mid x_{<t})\right) = \exp(\text{loss})
$$

Cross-entropy loss 의 지수승. "평균적으로 모델이 다음 토큰 후보 몇 개 사이에서 헷갈리는가" 의 직관.

| Loss | PPL | 의미 |
|---:|---:|---|
| ln(8000) ≈ 8.99 | 8000 | 무작위 (모든 토큰 동등) |
| 2.45 | 11.6 | 본 책 10M 모델 (Ch 15) |
| 2.0 | 7.4 | TinyStories 33M |
| 1.5 | 4.5 | GPT-2 (124M) on WebText |
| 1.0 | 2.7 | 큰 모델 (7B) on 일반 텍스트 |

## 2. PPL 만으로 안 되는 4가지 함정

1. **토크나이저가 다르면 비교 불가** — 같은 텍스트도 토큰 수 다름 → per-character 환산 필요
2. **도메인 분포 차이** — 본 책 10M 동화 모델의 Wikipedia PPL = 1000+
3. **PPL 낮은데 출력은 엉망** — 긴 시퀀스 일관성·논리·사실은 반영 안 함
4. **PPL 의 한계 effective** — 70B+ 에서 PPL 차이 0.1 미만이지만 능력 차이는 명확

## 4. 최소 예제 — PPL 측정 5줄

In [ ]:
# measure_ppl.py
import math, torch
# from nano_gpt import GPTMini, GPTConfig  # Ch 10 에서 정의한 모델

@torch.no_grad()
def perplexity(model, val_loader, device='cuda'):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        total_loss += loss.item() * y.numel()                         # 토큰 수 가중치
        total_tokens += y.numel()
    return math.exp(total_loss / total_tokens)                         # PPL = exp(평균 loss)

# 사용
# ppl = perplexity(model, val_loader)
# print(f"PPL: {ppl:.2f}")

# loss -> PPL 변환 직접 확인
print("loss -> PPL 변환:")
for loss_val in [8.99, 2.456, 2.0, 1.5, 1.0]:
    print(f"  loss={loss_val:.3f}  PPL={math.exp(loss_val):.1f}")

print()
print("본 책 10M 동화 모델 (val 1M 토큰):")
print("  PPL: 11.65   (loss 2.456)")
print("  해석: 다음 토큰을 약 12개 후보 안에서 골라야 한다 (vocab 8K 중 1/700)")

## 5. 실전 — 생성 샘플 검토 프로토콜

### 5.1 카테고리 분류

| 카테고리 | prompt | 평가 항목 |
|---|---|---|
| 인물 등장 | "Once upon a time, there was" | 자연스러운 인물·이름 |
| 사물 발견 | "Lily found a" | 합리적 사물 |
| 감정 표현 | "The dog was very" | 감정 표현 적절 |
| 대화 | "She said," | 대화 형식 |
| 마무리 | "...and they all lived" | "happily ever after" 정형 |

### 5.2 블라인드 평가

In [ ]:
# blind_eval.py
import random, json

prompts = [
    "Once upon a time, there was",
    "Lily found a",
    "The dog was very",
    'She said,',
    "...and they all lived",
]

def mock_generate(prompt, model_name):
    """실제 모델이 없을 때 데모용 더미 생성."""
    samples = {
        "model_a": {
            "Once upon a time, there was": " a little girl named Lily.",
            "Lily found a": " flower in the garden.",
            "The dog was very": " happy and wagged his tail.",
            'She said,': ' "Let us go to the park!"',
            "...and they all lived": " happily ever after.",
        },
        "model_b": {
            "Once upon a time, there was": " a very big tree in the forest.",
            "Lily found a": " small puppy under the bench.",
            "The dog was very": " sad because it was raining.",
            'She said,': ' "I want to play outside."',
            "...and they all lived": " happily in the cozy house.",
        },
    }
    return prompt + samples.get(model_name, {}).get(prompt, " ...")

# 두 모델로 생성
samples = []
for p in prompts:
    a = mock_generate(p, "model_a")
    b = mock_generate(p, "model_b")
    flip = random.random() < 0.5                                       # 랜덤 좌/우 배치
    samples.append({"prompt": p, "left": a if flip else b, "right": b if flip else a, "is_a_left": flip})

# 샘플 출력 (실제론 input() 으로 라벨링)
for s in samples:
    print(f"\nPROMPT: {s['prompt']}")
    print(f"  LEFT:  {s['left']}")
    print(f"  RIGHT: {s['right']}")

print("\n[실제 평가 시 input('Better (L/R/T): ') 으로 라벨링]")

# 집계 -- A 의 승률
# a_wins = sum(1 for s in samples if (s['choice'] == 'L') == s['is_a_left'])
# ties = sum(1 for s in samples if s['choice'] == 'T')
# print(f"A 승: {a_wins}, B 승: {len(samples)-a_wins-ties}, 무승부: {ties}")

In [ ]:
# 5축 평가 점수 요약 (본 책 10M 모델 평균)
scores = {
    "문법": 4.6,
    "일관성": 3.4,
    "어휘": 4.5,
    "창의성": 2.9,
    "마무리": 2.8,
}

print("5축 평가 점수 (0~5)")
print("-" * 35)
for axis, score in scores.items():
    filled = int(score)
    bar = "#" * filled + "." * (5 - filled)
    print(f"  {axis:5s}  [{bar}]  {score:.1f}")
print(f"\n  평균: {sum(scores.values())/len(scores):.2f}")
print("\n  약점: 창의성·마무리 -> 더 큰 모델 또는 다양한 데이터 필요")

## 연습문제

1. 본 책 모델로 val PPL 을 측정하고 학습 마지막 step 의 loss 와 일치하는지 확인 (`exp(loss)`).
2. **OOD PPL 측정** — Wikipedia 영어 단편 1000 토큰을 입력. PPL 이 얼마? hold-out PPL 과의 차이는?
3. 50 prompt × 5 축 블라인드 평가를 본인이 직접 수행. 가장 약한 축은?
4. **temperature 0.0 / 0.5 / 1.0 / 1.5** 로 같은 prompt 5개 생성. PPL 은 같은데 다양성·정확도가 어떻게 변하나?
5. **(생각해볼 것)** PPL 이 학습 토큰 수와 함께 떨어지지 않는 (saturate 하는) 시점은? 그 시점에서 모델은 무엇을 더 배울 수 있나?

In [ ]:
# 연습 1: loss -> PPL 확인


In [ ]:
# 연습 2: OOD PPL 측정


In [ ]:
# 연습 3: 50 prompt 블라인드 평가


In [ ]:
# 연습 4: temperature 별 생성 비교


In [ ]:
# 연습 5: PPL saturation 관찰
